## Baseline Model

### Training of the Model

In [6]:
import torch
from model import SimpleCNN

# 1. Initialize our model
model = SimpleCNN(num_classes=8)

# 2. Create a "dummy" batch of images (Batch Size=1, Channels=3, Height=224, Width=224)
# We fill it with random numbers to simulate image pixels
dummy_images = torch.randn(1, 3, 224, 224)

# 3. Pass it through the model (this automatically calls the `forward` function)
output = model(dummy_images)

print("Model Output Shape:", output.shape)
print("Raw Scores for the 8 classes:\n", output.detach().numpy())

Model Output Shape: torch.Size([1, 8])
Raw Scores for the 8 classes:
 [[ 0.08735376  0.02155809  0.00075822  0.0228718   0.10331789 -0.05877749
   0.0258783   0.01993049]]


### Creating a Log Book to keep history of model accuracy

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import csv
import os
from datetime import datetime

In [ ]:
def log_experiment(model_name, epochs, batch_size, learning_rate, 
                   train_acc, val_acc, notes, log_file='./modelAcc_history.csv'):
    file_exists = os.path.isfile(log_file)
    with open(log_file, mode='a', newline='') as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(['Timestamp', 'Model_Name', 'Epochs', 'Batch_Size', 
                             'Learning_Rate', 'Train_Accuracy', 'Val_Accuracy', 'Notes'])
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        writer.writerow([timestamp, model_name, epochs, batch_size, 
                         learning_rate, f"{train_acc:.4f}", f"{val_acc:.4f}", notes])

### Model Train Function

In [3]:
def train_model(model, train_loader, val_loader, criterion, optimizer, model_name, num_epochs=10):
    # Check if a GPU is available, otherwise use CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")
    model.to(device)
    
    best_val_acc = 0.0

    for epoch in range(num_epochs):
        # --- 1. TRAINING PHASE ---
        model.train() # Tells layers like Dropout to turn ON
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Step 1: Clear old gradients
            optimizer.zero_grad()
            
            # Step 2: Forward pass (guess the class)
            outputs = model(images)
            
            # Step 3: Calculate the error
            loss = criterion(outputs, labels)
            
            # Step 4: Backward pass (calculate the adjustments)
            loss.backward()
            
            # Step 5: Update the weights
            optimizer.step()
            
            # Track metrics
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
        train_acc = correct_train / total_train
        
        # --- 2. VALIDATION PHASE ---
        model.eval() # Tells layers like Dropout to turn OFF (we want full power for testing)
        correct_val = 0
        total_val = 0
        val_loss = 0.0
        
        # Checking accuracy with validation is 20% of the train data
        with torch.no_grad(): # Don't calculate gradients (saves memory and speeds up testing)
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                
        val_acc = correct_val / total_val
        
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {running_loss/len(train_loader):.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.4f}")
        
        # Save the best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            os.makedirs('./outputs', exist_ok=True)
            torch.save(model.state_dict(), model_name)
            
    return train_acc, best_val_acc

### Loading the Data

### Connecting to Drive

In [ ]:
from datacleaner import build_training_dataloaders

train_loader, val_loader = build_training_dataloaders(
    csv_path='./raw_data/train_labels_strId.csv', 
    img_dir='./raw_data/train/train', 
    batch_size=32
)

### Execution

In [ ]:
# 1. Bring in our model
from model import SimpleCNN
model = SimpleCNN(num_classes=8)

# 2. Define hyper-parameters
LEARNING_RATE = 0.001
EPOCHS = 5 # Keep it small just to test that it runs without crashing
BATCH_SIZE = 32

# 3. Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 4. RUN IT!
print("Starting training...")
final_train_acc, final_val_acc = train_model(
    model, train_loader, val_loader, criterion, optimizer, 
    './baseline_model_v1.pth',
    num_epochs=EPOCHS
)

# 5. Log the experiment
log_experiment(
    model_name="SimpleCNN_Baseline",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    train_acc=final_train_acc,
    val_acc=final_val_acc,
    notes="Baseline Model V.1."
)

Starting training...
Training on device: cuda
Epoch [1/5] Train Loss: 0.6858, Train Acc: 0.7590 | Val Loss: 0.2332, Val Acc: 0.9278
Epoch [2/5] Train Loss: 0.2731, Train Acc: 0.9094 | Val Loss: 0.1653, Val Acc: 0.9445
Epoch [3/5] Train Loss: 0.2215, Train Acc: 0.9279 | Val Loss: 0.1405, Val Acc: 0.9533
Epoch [4/5] Train Loss: 0.1876, Train Acc: 0.9400 | Val Loss: 0.1325, Val Acc: 0.9561
Epoch [5/5] Train Loss: 0.1750, Train Acc: 0.9424 | Val Loss: 0.1247, Val Acc: 0.9589


## Transfer Model
The training was done on Colab, just the code here

In [ ]:
# 1. Bring in our model
from transfermodel import TransferResNet
model = TransferResNet(num_classes=8)
import torch.optim as optim

# 2. Define hyper-parameters
LEARNING_RATE = 0.001
EPOCHS = 5 # Keep it small just to test that it runs without crashing
BATCH_SIZE = 32

# 3. Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()
# This filters out the frozen layers and only passes the new Linear layer to the optimizer
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.Adam(trainable_params, lr=LEARNING_RATE)

# 4. RUN IT!
print("Starting training...")
final_train_acc, final_val_acc = train_model(
    model, train_loader, val_loader, criterion, optimizer, 
    './outputs/transfer_model_v1.pth',
    num_epochs=EPOCHS
)

# 5. Log the experiment
log_experiment(
    model_name="Transfer_Model_v1",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    train_acc=final_train_acc,
    val_acc=final_val_acc,
    notes="Transfer Model using Pre Trained Model."
)

## Transfer Model Tuned

### Importaning our "BaseLine" Transfer Model

In [2]:
import torch
import torch.nn as nn
from torchvision import models

In [ ]:
from transfermodel import TransferResNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransferResNet(num_classes=8)

# Loading our Model
model.load_state_dict(torch.load('./outputs/transfer_model_v1.pth', map_location=device))
model.to(device)

TransferResNet(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-0

In [ ]:
for param in model.parameters():
    param.requires_grad = True

FINE_TUNE_LR = 0.0001
optimizer = optim.Adam(model.parameters(), lr=FINE_TUNE_LR)
criterion = torch.nn.CrossEntropyLoss()

# 4. Run the Training Loop for a few more epochs
print("Starting Fine-Tuning Phase...")
FINE_TUNE_EPOCHS = 5

final_train_acc, final_val_acc = train_model(
    model, 
    train_loader, 
    val_loader, 
    criterion, 
    optimizer, 
    './outputs/transfer_model_v2.pth',
    num_epochs=FINE_TUNE_EPOCHS
)

### Saving our Accuracy Data

In [ ]:
# 6. Log the experiment
log_experiment(
    model_name="ResNet18_FineTuned",
    epochs=FINE_TUNE_EPOCHS,
    batch_size=BATCH_SIZE, # (Assuming this is still 32 in your memory)
    learning_rate=FINE_TUNE_LR,
    train_acc=final_train_acc,
    val_acc=final_val_acc,
    notes="Unfroze all layers and fine-tuned with a lower learning rate."
)

## Transfer Model Fine Tuned

In [ ]:
train_loader, val_loader = build_training_dataloaders(
    csv_path='./KaggleCompetition/raw_data/train_labels_strId.csv', 
    img_dir='./KaggleCompetition/raw_data/train/train', 
    batch_size=32
)

### Importing the Model

In [ ]:
# 2. Setup the Unfrozen ResNet50 Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training ultimate model on device: {device}")

model = AdvancedTransferModel(num_classes=8, freeze_features=False)
model.to(device)

### Setting our Model Settings

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
# The scheduler will cut the learning rate in half if Val Acc doesn't improve for 2 epochs
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# 4. The Complete Training Loop
# Increase the epochs
num_epochs = 12
best_val_acc = 0.0
save_path = './KaggleCompetition/transfer_model_v3.pth'

### Training our Model

In [ ]:
print("Starting Ultimate Fine-Tuning Phase...")
for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train() 
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    train_acc = correct_train / total_train
    
    # --- VALIDATION PHASE ---
    model.eval() 
    correct_val = 0
    total_val = 0
    val_loss = 0.0
    
    with torch.no_grad(): 
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    val_acc = correct_val / total_val
    
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.4f}")
    
    # Pass the Validation Accuracy to the Scheduler so it knows when to drop the learning rate
    scheduler.step(val_acc)
    
    # --- SAVE THE BEST MODEL ---
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        os.makedirs('./outputs', exist_ok=True),
        torch.save(model.state_dict(), save_path)
        print(f"--> New High Score! Model saved to {save_path}")

print("Ultimate training complete.")

### Save the Data

In [ ]:
# 5. Log the Ultimate Experiment
log_experiment(
    model_name="ResNet50_Ultimate_120x120",
    epochs=num_epochs,
    batch_size=32, 
    learning_rate=0.0001, # The starting learning rate
    train_acc=train_acc,  # Final training accuracy
    val_acc=best_val_acc, # The highest validation accuracy achieved
    notes="Ultimate pipeline: ResNet50, 120x120 crop, hard augmentations, fully unfrozen, ReduceLROnPlateau scheduler."
)

print("Experiment successfully saved to history!")

## Transfer Model Fine Tuned V.2

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from datacleaner_v2 import build_training_dataloaders
from advancemodel import AdvancedTransferModel

### Loading the Data

In [4]:
# 1. Load the Data (Using 120x120 transforms in your data_utils)
train_loader, val_loader = build_training_dataloaders(
    csv_path='./raw_data/train_labels.csv', 
    img_dir='./raw_data/train/train', 
    batch_size=32
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

### Phase 1: Training with Freeze ON

In [ ]:
print("--- PHASE 1: Training Frozen Head ---")
# Initialize WITH frozen features to protect ResNet50
model = AdvancedTransferModel(num_classes=8, freeze_features=True)
model.to(device)

# Pass ONLY the unfrozen final layer to the optimizer
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer_phase1 = optim.Adam(trainable_params, lr=0.001)

# Run a short training loop to stabilize the random weights
train_acc_p1, val_acc_p1 = train_model(model, 
    train_loader, 
    val_loader, 
    criterion, 
    optimizer_phase1, 
    '/transfer_model_v4_1.pth',
    num_epochs=5
)

### Phase 2: Training

In [ ]:
print("\n--- PHASE 2: Unfreezing and Fine-Tuning ---")
# 1. Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

# 2. Use a MICROSCOPIC learning rate to protect the stabilized weights
optimizer_phase2 = optim.Adam(model.parameters(), lr=0.00001) 
scheduler = ReduceLROnPlateau(optimizer_phase2, mode='max', factor=0.5, patience=2)

# 3. Custom Loop for Phase 2 to integrate the Scheduler
num_epochs_phase2 = 10
best_val_acc = val_acc_p1
save_path = './outputs/trasfer_model_v4_1.pth'

for epoch in range(num_epochs_phase2):
    # --- TRAINING ---
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_phase2.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_phase2.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
    train_acc = correct_train / total_train
    
    # --- VALIDATION ---
    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
    val_acc = correct_val / total_val
    
    print(f"Epoch [{epoch+1}/{num_epochs_phase2}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    
    # Step the scheduler based on validation accuracy
    scheduler.step(val_acc)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"--> New High Score! Model saved to {save_path}")

print("Ultimate training complete.")

In [ ]:
log_experiment(
    model_name="Transfer Model 50 Layers Trained in 2 Phase",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    train_acc=final_train_acc,
    val_acc=final_val_acc,
    notes="Transfer Model: Combination of Freezer ON and OFF"
)

## Back to 18 Layers more Fine Tuned

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from datacleaner_v2 import build_training_dataloaders
from transfermodel import TransferResNet

### Log Function

In [4]:
def log_experiment(model_name, epochs, batch_size, learning_rate, 
                   train_acc, val_acc, notes, log_file='./modelAcc_history.csv'):
    file_exists = os.path.isfile(log_file)
    with open(log_file, mode='a', newline='') as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(['Timestamp', 'Model_Name', 'Epochs', 'Batch_Size', 
                             'Learning_Rate', 'Train_Accuracy', 'Val_Accuracy', 'Notes'])
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        writer.writerow([timestamp, model_name, epochs, batch_size, 
                         learning_rate, f"{train_acc:.4f}", f"{val_acc:.4f}", notes])

### Training Function

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, model_name, num_epochs=10):
    # Check if a GPU is available, otherwise use CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")
    model.to(device)
    
    best_val_acc = 0.0

    for epoch in range(num_epochs):
        # --- 1. TRAINING PHASE ---
        model.train() # Tells layers like Dropout to turn ON
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Step 1: Clear old gradients
            optimizer.zero_grad()
            
            # Step 2: Forward pass (guess the class)
            outputs = model(images)
            
            # Step 3: Calculate the error
            loss = criterion(outputs, labels)
            
            # Step 4: Backward pass (calculate the adjustments)
            loss.backward()
            
            # Step 5: Update the weights
            optimizer.step()
            
            # Track metrics
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
        train_acc = correct_train / total_train
        
        # --- 2. VALIDATION PHASE ---
        model.eval() # Tells layers like Dropout to turn OFF (we want full power for testing)
        correct_val = 0
        total_val = 0
        val_loss = 0.0
        
        # Checking accuracy with validation is 20% of the train data
        with torch.no_grad(): # Don't calculate gradients (saves memory and speeds up testing)
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                
        val_acc = correct_val / total_val
        
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {running_loss/len(train_loader):.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.4f}")
        
        # Save the best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            os.makedirs('./outputs', exist_ok=True)
            torch.save(model.state_dict(), model_name)
            
    return train_acc, best_val_acc

In [ ]:
train_loader, val_loader = build_training_dataloaders(
    csv_path='./raw_data/train_labels.csv', 
    img_dir='./raw_data/train/train', 
    batch_size=32
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()
epochs = 4

### PHASE 1: STABILIZE THE HEAD (FROZEN)

In [3]:
print("--- PHASE 1: Training Frozen Head ---")
model = TransferResNet(num_classes=8, freeze_features=True).to(device)
optimizer_phase1 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

--- PHASE 1: Training Frozen Head ---


In [ ]:
train_acc, val_acc = train_model(
    model, 
    train_loader, 
    val_loader, 
    criterion, 
    optimizer_phase1, epochs, 
    './outputs/transfer_model__v4_phase1.pth'
)

In [ ]:
# 5. Log the experiment
log_experiment(
    model_name="Transfer_Model_ResNet18_v2",
    epochs=4,
    batch_size=32,
    learning_rate=0.001,
    train_acc=final_train_acc,
    val_acc=final_val_acc,
    notes="Transfer Model ResNet18 tuned phase 1 FROZEN."
)

### Phase 2: UnFrozen

In [ ]:
print("\n--- PHASE 2: Unfreezing and Fine-Tuning ---")
for param in model.parameters():
    param.requires_grad = True

num_epochs_phase2 = 15

# Added weight_decay (L2 Regularization) to prevent overfitting
optimizer_phase2 = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
# Added CosineAnnealing to smoothly curve the learning rate downward
scheduler = CosineAnnealingLR(optimizer_phase2, T_max=num_epochs_phase2)

best_val_acc = val_acc_p1
save_path = './outputs/resnet18_optimal.pth'

for epoch in range(num_epochs_phase2):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_phase2.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer_phase2.step()
        
    model.eval()
    correct_val, total_val = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            _, predicted = torch.max(model(images).data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    val_acc = correct_val / total_val
    print(f"Epoch [{epoch+1}/{num_epochs_phase2}] Val Acc: {val_acc:.4f}")
    
    # Step the cosine scheduler (does not need val_acc passed in)
    scheduler.step()
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"--> New High Score! Saved to {save_path}")